<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - Reverse Engineering</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 1: Static RE</h1>

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">1 - What is static reversing?</h1>

Static reversing is the process of comprehending a binary without actually having to run it. You've seen a very basic version of this in Intro to RE, but today we will see the tools that allow for significantly more control over the process.

Tha tool we will be using today is called **Ghidra**

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - Intro to Ghidra</h1>

Navigate to the [Ghidra page](https://github.com/NationalSecurityAgency/ghidra), and scroll down to the **Install** section. Follow the instructions there to download and run Ghidra.

After installing and launching Ghidra, you should see something that looks like\
<img src="./media/ghidra-projects.png">

Once here, you're going to go to the File tab, and select "New Project". Create a project in a location of your choice(I'd recommend keeping it in the same place as your binaries). Next, import a binary(File -> Import File...), we'll start with `chal1` from Intro to RE. Your Ghidra project should look something like\
<img src="./media/chal1-ghidra.png">

Double click on the imported file, and it should open the Code Browser. You should see this\
<img src="./media/analyze.png">

Select "Yes". This is the power of Ghidra, it turns that messy binary back into code. You can also do this manually by selecting **Analysis** -> **Auto Analyze ...** in the top left. This might take a bit depending on your hardware and the size of the binary you analyze. If you're following along, `chal1` is pretty small, so it shouldn't take long. In the top left, you should see the sections of the program. The most important sections being the `.data` section, where global variables are defined, and `.text`, where the code execution occurs. You can also find there if you go to **Window** -> **Memory Map**.

The memory map displays all of the different sections in the code. That is, the binary has data telling the computer where different things are, and Ghidra has reconstructed them for you. Ghidra reconstructs other data as well, such as the symbol tree(left), which will list all functions, labels, classes, etc., the listing(center) showing the assembly, plus some reconstructed data, the decompiler(right), showing the code representation

If you go to the `.text` section(double click), you'll see what looks like a lot of nonsense. This is the libc entrypoint, not where our code begins. To find that, press g and type in "main". This is the "Go To" command. If everything worked right, you should be able to see the decompilation for main. See how it looks like code? And look at that, you can see the flag from `chal1`!

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.1 - What are the weird variables?</h2>

Ghidra automatically generates variable names, but Ghidra doesn't know what they were originally, so it just uses filler like `local_8`. For more difficult challenges, it's helpful to change this. Select a variable and press "l", or right click and select "Rename Variable". Ghidra will immediately change all instances to whatever you set it to. Additoinally, you can change the type of variables and functions(press "CTRL+L" or right click -> retype variable).

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Practice Challenge</h1>

Try to do the same with `chal2` and see what you can find.

Tip for challenges: try to fin the "win function". This is the part in the code where the program dumps the flag or opens the shell.(It's not always a function, but in this case it is) Once you know "how to win", work backward to the beginning of the program or where your interaction starts.

Did you find the system call? Work backward and see if you can match the parts of decompiled code to the previous Intro to RE breakdown\
<img src="./media/chal2-annotated.png">

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">3 - Sublime Text</h1>

Now we are going to try to remove the annoying popup for Sublime Text\
<img src="./media/popup.png">

Unzip the apporpriate version of Sublime(.zip for Windows, .tar.xz for Linux). Then, create a new project and import the executable. After analyzing, you might notice an issue compared to what we did last time:\
<img src="./media/no_main.png">

This is what is known as a *stripped binary*. This means that Ghidra wasn't able to reconstruct the labels for the data, so we'll have to find main ourselves. To start, lets look around. As seen on the previous image, there is a function called `entry()`, which seems like a good place to start. Search for `entry`(using the "Go To" command, **g**). On the decompilation, you will find a call for `__libc_start_main`. Looking at it, notice that the first argument is a function parameter. Lets double click that to follow it. Behold! Our `main` function!

Try to find where the box popup is located in the file. **Window** -> **Defined Strings** can be used to find text, try matching the popup text. **Right click** -> **References** can be used to find where it's used)\
Once you've found the popup code, get rid of it using patching:

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">3.1 - Patching</h2>

**Right click** -> **Patch instruction** allows you to enter assembly into the code by changing the binary data. Write some assembly that avoids the popup! Remember to create a backup so you can revert your changes if you mess up.(Hint: try to override the function that creates the popup or the `if` statement that calls it)

Once you create the patch, save the program and try running Sublime Text. You'll notice now that no matter how much you try, the popup no longer appears!